# Проект: Анализ поведения пользователей для повышения удержания клиентов



Автор проекта:  Чанышев Исмагил
Дата: 04.03.2026

## Введение

Команда маркетинга стремится лучше понимать поведение пользователей платформы с целью повышения уровня удержания клиентов. Для этого проводится исследовательский анализ данных, который позволит выявить закономерности в поведении пользователей и определить, какие факторы влияют на вероятность возврата клиентов и совершения повторных заказов.

# Оглавление



### Цели проекта

1. Выявить пользователей с высокой вероятностью возврата на платформу
2. Определить признаки первого заказа, влияющие на повторные покупки
3. Проанализировать связь между метриками заказа (выручка, количество билетов) и удержанием клиентов
4. Сформулировать рекомендации для оптимизации маркетинговых бюджетов

### Задачи проекта

- Загрузка и предобработка данных из базы данных SQL
- Создание профиля пользователя с агрегированными признаками
- Исследовательский анализ данных (EDA)
- Корреляционный анализ признаков
- Формулировка выводов и рекомендаций



Выгрузка из базы данных SQL позволила собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ ( `mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk` );
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk` );
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.



## Шаг 1. Загрузка данных

**Цель шага:** Получить исходные данные для анализа из базы данных и провести первичную оценку их качества.

Выгружаем данные о заказах пользователей, включая информацию об устройствах, выручке, мероприятиях и временных интервалах между покупками. Фильтруем данные, исключая фильмы и оставляя только заказы с мобильных и десктопных устройств.

In [12]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
from phik import phik_matrix
from dotenv import load_dotenv
import os

Задача 1.1: Напишем SQL-запрос, выгружающий в датафрейм pandas необходимые данные.

In [5]:
load_dotenv()

db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME'),
}

In [6]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

In [7]:
engine = create_engine(connection_string)

In [8]:
query = '''
SELECT user_id,
    device_type_canonical,
    order_id,
    created_dt_msk::DATE AS order_dt, 
    created_ts_msk AS order_ts,
    currency_code,
    revenue,
    tickets_count,
    created_dt_msk - LAG(created_dt_msk) OVER (PARTITION BY user_id ORDER BY created_dt_msk) AS days_since_prev,
    event_id,
    event_name_code AS event_name,
    service_name,
    event_type_main,
    region_name,
    city_name
FROM afisha.purchases AS p
JOIN afisha.events AS e USING(event_id)
JOIN afisha.city AS c USING(city_id)
JOIN afisha.regions AS r USING(region_id)
WHERE (device_type_canonical = 'desktop' OR device_type_canonical = 'mobile' ) AND NOT event_type_main ='фильм'
ORDER BY user_id
''' 

In [9]:
df = pd.read_sql_query(query, con=engine)

In [10]:
df.head()


,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,service_name,event_type_main,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaT,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,Край билетов,театр,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaT,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,Мой билет,выставки,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75 days,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,За билетом!,другое,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaT,387271,2f638715-8844-466c-b43f-378a627c419f,Лови билет!,другое,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83 days,509453,10d805d3-9809-4d8a-834e-225b7d03f95d,Билеты без проблем,стендап,Озернинский край,Родниковецк


In [11]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype          
---  ------                 --------------   -----          
 0   user_id                290611 non-null  object         
 1   device_type_canonical  290611 non-null  object         
 2   order_id               290611 non-null  int64          
 3   order_dt               290611 non-null  object         
 4   order_ts               290611 non-null  datetime64[ns] 
 5   currency_code          290611 non-null  object         
 6   revenue                290611 non-null  float64        
 7   tickets_count          290611 non-null  int64          
 8   days_since_prev        268678 non-null  timedelta64[ns]
 9   event_id               290611 non-null  int64          
 10  event_name             290611 non-null  object         
 11  service_name           290611 non-null  object         
 12  event_type_main        290611 

### Промежуточный вывод по Шагу 1

- Получено 290 611 записей о заказах
- Данные содержат 15 столбцов с информацией о пользователях, заказах и мероприятиях
- Столбец `days_since_prev` содержит пропуски для пользователей с одним заказом (ожидаемое поведение)
- Требуется предобработка: конвертация валют, исправление типов данных, фильтрация выбросов
- order_dt можно перевести в формат даты
- tickets_count можно понизить разрядность

# Шаг 2. Предобработка данных

**Цель шага:** Подготовить данные к анализу, обеспечив их корректность и сопоставимость.

загрузка информации с курсом казахского тенге. Значения в рублях представлено для 100 тенге.

In [ ]:
df_tenge = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [ ]:
df_tenge

Видно, что строк 356. проверим только, чтобы не повторялись значения дней

In [ ]:
df_tenge['data'].duplicated().sum()

In [ ]:
df_tenge.info()

добавим в исходную таблицу с данными столбцы nominal и curs из таблицы с курсом валют. объединяем по столбцам order_dt и data, предварительно преобразовав тип данных в datetime, чтобы исключить ошибки

In [ ]:
df['order_dt'] = pd.to_datetime(df['order_dt'], format='%Y-%m-%d')
df_tenge['data'] = pd.to_datetime(df_tenge['data'], format='%Y-%m-%d')

In [ ]:
df = df.merge(df_tenge, how = 'left', left_on = 'order_dt', right_on ='data')

In [ ]:
df.head()

добавим столбец `revenue_rub` и удалим лишние столбцы

In [ ]:
# Копируем исходную сумму
df['revenue_rub'] = df['revenue']

# Для строк с тенге пересчитываем
mask_kzt = df['currency_code'] == 'kzt'
df.loc[mask_kzt, 'revenue_rub'] = df.loc[mask_kzt, 'revenue'] * df.loc[mask_kzt, 'curs'] / 100

In [ ]:
del df['data']
del df['nominal']
del df['curs']
del df['cdx']

In [ ]:
df.head()


**Задача 2.2.**

- Проверим данные на пропущенные значения.
- Преобразуем типы данных в некоторых столбцах, если это необходимо. 
- Изучим значения в ключевых столбцах. 
    - Проверим, какие категории указаны в столбцах с номинальными данными. 
    - Проверим распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха. Важные показатели в рамках поставленной задачи — это выручка с заказа ( `revenue_rub` ) и количество билетов в заказе ( `tickets_count` ), поэтому в первую очередь проверим данные в этих столбцах. Если обнаружим выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.



In [ ]:
tmp = df # сохраним копию датасета для подсчета того, что отфильтруем
df.info()

In [ ]:
# понизим размерность столбца tickets_count
df['tickets_count'] = pd.to_numeric(df['tickets_count'], downcast = 'integer')

изучим столбцы с категориальными данными на предмет неявных дубликатов и значений, обозначающих пропуски

In [ ]:
# изучим столбец device_type_canonical
df['device_type_canonical'].unique()

In [ ]:
# изучим столбец currency_code
df['currency_code'].unique()

In [ ]:
# изучим столбец service_name
df['service_name'].unique()

In [ ]:
# изучим столбец event_type_main
df['event_type_main'].unique()

In [ ]:
# изучим столбец region_name
df['region_name'].sort_values().unique()

In [ ]:
# изучим столбец city_name
df['city_name'].sort_values().unique()

Проверим распределение численных данных и наличие в них выбросов

столбец revenue_rub

In [ ]:
df['revenue_rub'].describe()

In [ ]:
# нарисуем диаграмму , отфильтровав 99-процентиль, т.к. большие выбросы
df[(df['revenue_rub'] < df['revenue_rub'].quantile(0.99))].hist(column = 'revenue_rub', bins = 50, figsize = (12,8))

In [ ]:
df.boxplot(column = 'revenue_rub', vert=False, figsize = (12,4))

отфильтруем выбросы

In [ ]:
df.boxplot(column = 'revenue_rub', vert=False, figsize = (12,4), showfliers= False)

сделаем то же с tickets_count

In [ ]:
df['tickets_count'].describe()

In [ ]:
# нарисуем диаграмму , отфильтровав 99-процентиль, т.к. большие выбросы
df[(df['tickets_count'] < df['tickets_count'].quantile(0.99))].hist(column = 'tickets_count', bins = 6, figsize = (10,8))

In [ ]:
df.boxplot(column = 'tickets_count', vert=False, figsize = (12,4))

In [ ]:
df.boxplot(column = 'tickets_count', vert=False, figsize = (12,4), showfliers= False)

Оцените, в каком объёме были отфильрованы данные. 

In [ ]:
# для столбца revenue_rub были отфильтрованы данные
rows_filtered = df[(df['revenue_rub'] < df['revenue_rub'].quantile(0.99))].shape[0]
total = df.shape[0]
filtered_perc = round(rows_filtered / total, 2)
print(f'Для графиков "revenue_rub" было отфильтровано {rows_filtered} строк из {total}, {filtered_perc}%.')

# для столбца tickets_count были отфильтрованы данные
rows_filtered = df[(df['tickets_count'] < df['tickets_count'].quantile(0.99))].shape[0]
total = df.shape[0]
filtered_perc = round(rows_filtered / total, 2)
print(f'Для графиков "tickets_count" было отфильтровано {rows_filtered} строк из {total}, {filtered_perc}%.')

df = df[(df['tickets_count'] < df['tickets_count'].quantile(0.99))]
df = df[(df['revenue_rub'] < df['revenue_rub'].quantile(0.99))]

print()
print(f'Всего отфильтровано {df.shape[0]} строк из {tmp.shape[0]}, что составляет {round(df.shape[0] / tmp.shape[0] *100, 2)}%')

In [ ]:
df.shape[0]

### Что было сделано:

1. **Конвертация валют:** Добавлен курс казахского тенге, все значения приведены к рублям через столбец `revenue_rub`
2. **Преобразование типов данных:**
    - `order_dt` преобразован в datetime
    - `tickets_count` понижена разрядность
3. **Проверка на пропуски:** Пропуски обнаружены только в столбце `days_since_prev` (21 933 значений)
4. **Фильтрация выбросов:**
    - По `revenue_rub` отфильтровано 1% данных (99-й перцентиль)
    - По `tickets_count` отфильтровано 2% данных (99-й перцентиль)

### Промежуточный вывод по Шагу 2

- Данные приведены к единой валюте (рубли)
- Выявлены и отфильтрованы статистические выбросы по выручке и количеству билетов
- Категориальные данные проверены на наличие неявных дубликатов
- Dataset готов для агрегации и создания профилей пользователей

---

## Шаг 3. Создание профиля пользователя

**Цель шага:** Сформировать агрегированные признаки для каждого пользователя, описывающие его поведение и историю заказов.


**Задача 3.1.** Построим профиль пользователя — для каждого пользователя найдём:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используем поле `event_type_main` );
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавим два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.


In [ ]:
df_users = df.groupby(by='user_id').agg(
    first_order_dt = ('order_dt', 'min'),
    last_order_dt = ('order_dt', 'max'),
    first_order_device = ('device_type_canonical', 'first'),
    first_order_region = ('region_name', 'first'),
    first_order_service = ('service_name', 'first'),
    first_order_event = ('event_type_main', 'first'),
    total_orders = ('order_id', 'count'),
    revenue_mean = ('revenue_rub', 'mean'),
    total_tickets = ('tickets_count', 'sum')
).reset_index()
df_users['avg_tickets_in_order'] = df_users['total_tickets'] / df_users['total_orders']
del df_users['total_tickets']
df_users['avg_days_between_orders'] = (df_users['last_order_dt'] - df_users['first_order_dt']) / (df_users['total_orders'] - 1)

# напишем функцию, определяющую бинарные признаки
def is_two(row):
    return row['total_orders'] >= 2

def is_five(row):
    return row['total_orders'] >= 5
        
df_users['is_two'] = df_users.apply(is_two, axis = 1)
df_users['is_five'] = df_users.apply(is_five, axis = 1)
df_users

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными мы работаем: насколько они репрезентативны и нет ли в них аномалий.


Используя данные о профилях пользователей, рассчитаем:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

In [ ]:
# считаем общее число ползователей в выборке
users_count = df_users.shape[0]
# считаем среднюю выручку с одного заказа
avg_revenue = round(df_users['revenue_mean'].mean(), 2)
#долю пользователей, совершивших 2 и более заказа
is_two_share = round(df_users['is_two'].mean(), 2)
#долю пользователей, совершивших 5 и более заказов
is_five_share = round(df_users['is_five'].mean(), 2)

print(f'общее число ползователей в выборке: {users_count}')
print(f'средняя выручка с одного заказа: {avg_revenue}')
print(f'доля пользователей, совершивших 2 и более заказа: {is_two_share}')
print(f'доля пользователей, совершивших 5 и более заказов: {is_five_share}')

Также изучим статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

In [ ]:
df_users[['total_orders', 'avg_tickets_in_order', 'avg_days_between_orders']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

# Статистика по выборке (после фильтрации):

- **Общее число пользователей:** 21 688
- **Средняя выручка с заказа:** 536 руб.
- **Доля пользователей с 2+ заказами:** 62%
- **Доля пользователей с 5+ заказами:** 29%

In [ ]:
df_users.boxplot(column = 'total_orders', vert=False, figsize = (12,4))

In [ ]:
df_users.boxplot(column = 'total_orders', vert=False, figsize = (12,4), showfliers = False)

In [ ]:
df_users[(df_users['total_orders'] < df_users['total_orders'].quantile(0.95))].hist(column = 'total_orders', bins = 30, figsize = (12,8))

отфильруем по 95-процентилю

In [ ]:
users_tmp = df_users
df_users = df_users[(df_users['total_orders'] < df_users['total_orders'].quantile(0.95))]
df_users[['total_orders', 'avg_tickets_in_order', 'avg_days_between_orders']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
filtered_rows = df_users.shape[0]

print(f'Отфильтровано {df_users.shape[0]} строк из {users_tmp.shape[0]}, {round(df_users.shape[0] / users_tmp.shape[0] *100, 2)}%')

### Промежуточный вывод по Шагу 3

- **Объём данных достаточен** для выявления закономерностей и сегментации пользователей.
- **Аномалии в `total_orders`** обусловлены наличием сверхактивных пользователей (возможно, корпоративные аккаунты или боты). Их фильтрация по 95-му перцентилю повышает надёжность анализа.
- **Данные по билетам** не требуют дополнительной обработки — распределение близко к нормальному с умеренным разбросом.
- После предобработки выборка из 20 580 пользователей готова к исследовательскому анализу.

---


## Шаг 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используем профили пользователей.

#### **4.1. Исследование признаков первого заказа и их связи с возвращением на платформу**

Исследуем признаки, описывающие первый заказ пользователя, и выясним, влияют ли они на вероятность возвращения пользователя.

**Задача 4.1.1.** Изучим распределение пользователей по признакам.

- Сгруппируем пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитаем общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. 
- Исследуем, равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей.

In [ ]:
df_users.head()


In [ ]:
# напишем функцию, считающую количество и долю клиентов в категории
def count_and_pct(df, column):
    counts = df[column].value_counts()
    pct = df[column].value_counts(normalize=True).round(3) * 100
    result = pd.DataFrame({
        'количество': counts,
        'доля_%': pct
    })
    return result

In [ ]:
# применим функцию к столбцу first_order_event
event_stats = count_and_pct(df_users, 'first_order_event')
event_stats

In [ ]:
# для сравнения по всем заказам, а не только по первому заказу
count_and_pct(df, 'event_type_main').head()

In [ ]:
# применим функцию к столбцу first_order_device
event_stats = count_and_pct(df_users, 'first_order_device')
event_stats

In [ ]:
# для сравнения по всем заказам, а не только по первому заказу
count_and_pct(df, 'device_type_canonical').head()

In [ ]:
# применим функцию к столбцу first_order_region
event_stats = count_and_pct(df_users, 'first_order_region')
event_stats

In [ ]:
# для сравнения по всем заказам, а не только по первому заказу
count_and_pct(df, 'region_name').head()

In [ ]:
# применим функцию к столбцу first_order_service
event_stats = count_and_pct(df_users, 'first_order_service')
event_stats

In [ ]:
# для сравнения по всем заказам, а не только по первому заказу
count_and_pct(df, 'service_name').head()

#### Вывод

Пользователи распределены **неравномерно** — выявлены сегменты с наибольшей долей пользователей. Однако при сравнении с распределением по всем заказам видно, что эта неравномерность отражает общую структуру платформы, а не специфические «точки входа».

**Сравнение распределения: первый заказ vs все заказы**

| Признак         | Сегмент            | Доля в первых заказах | Доля во всех заказах | Разница   |
| --------------- | ------------------ | --------------------- | -------------------- | --------- |
| Тип мероприятия | Концерты           | 44.4%                 | 39.6%                | +4.8 п.п. |
| Устройство      | Mobile             | 82.9%                 | 80.2%                | +2.7 п.п. |
| Регион          | Каменевский регион | 32.8%                 | 31.3%                | +1.5 п.п. |
| Оператор        | Билеты без проблем | 23.7%                 | 21.8%                | +1.9 п.п. |

1. **Неравномерность есть**, но она неспецифична: доминирующие сегменты в первых заказах почти совпадают с доминирующими сегментами по всем заказам.
2. **Вывод:** высокая доля пользователей в сегментах «концерты», «mobile», «Каменевский регион» обусловлена не тем, что эти каналы лучше привлекают новых клиентов, а тем, что они в принципе составляют основу трафика платформы.
3. **Практическое значение:**
    - Не стоит интерпретировать эти сегменты как «успешные точки входа» без дополнительного анализа конверсии.
    - Для выявления реальных точек роста нужно сравнивать не абсолютные доли, а **конверсию в повторный заказ** внутри каждого сегмента (что сделано в задаче 4.1.2).

---


**Задача 4.1.2.** Проанализируем возвраты пользователей:

- Для каждого сегмента вычислим долю пользователей, совершивших два и более заказа.
- Визуализируем результат подходящим графиком. Если сегментов слишком много, то поместим на график только 10 сегментов с наибольшим количеством пользователей. 
- Ответим на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?



In [ ]:
# считаем и визуализируем долю пользователей, совершивших два и более заказа в категории типа мероприятия
event_mean = df_users.groupby(by='first_order_event')['is_two'].agg(['count', 'mean']).sort_values(by='mean', ascending=False)
print(event_mean)
plt.figure(figsize=(10, 4))
event_mean['mean'].plot(kind='bar', legend=False, xlabel = 'Тип мероприятия', ylabel = 'Доля')
plt.title('Доля пользователей, совершивших два и более заказа')
plt.show()

In [ ]:
# считаем и визуализируем долю пользователей, совершивших два и более заказа в категории типа устройства
device_mean = df_users.groupby(by='first_order_device')['is_two'].agg(['count', 'mean']).sort_values(by='mean', ascending=False)
print(device_mean)
plt.figure(figsize=(10, 4))
device_mean['mean'].plot(kind='bar', legend=False, xlabel = 'Тип устройства', ylabel = 'Доля')
plt.title('Доля пользователей, совершивших два и более заказа')
plt.show()

In [ ]:
# считаем и визуализируем долю пользователей, совершивших два и более заказа в категории регион
region_mean = df_users.groupby(by='first_order_region')['is_two'].agg(['count', 'mean']).sort_values(by='mean', ascending=False).head(10)
print(region_mean)
plt.figure(figsize=(10, 4))
region_mean['mean'].plot(kind='bar', legend=False, xlabel = 'Регион', ylabel = 'Доля')
plt.title('Доля пользователей, совершивших два и более заказа')
plt.show()

In [ ]:
# считаем и визуализируем долю пользователей, совершивших два и более заказа в категории оператора
service_mean = df_users.groupby(by='first_order_service')['is_two'].agg(['count', 'mean']).sort_values(by='mean', ascending=False).head(10)
print(service_mean)
plt.figure(figsize=(10, 4))
service_mean['mean'].plot(kind='bar', legend=False, xlabel = 'Оператор', ylabel = 'Доля')
plt.title('Доля пользователей, совершивших два и более заказа')
plt.show()

#### Вывод по задаче 4.1.2: Анализ возврата пользователей по сегментам


**Основные наблюдения:**
По типу мероприятия чаще возвращаются в сегментах : выставки (400 пользователей) - 63% и театр (4080 пользователей) - 0.62%  
По типу устройства чаще возвращаются в сегменте : desktop (3581 пользователей) - 62%  
По региону чаще возвращаются в сегментах : Озернопольская область (26 пользователей) - 88,5% и Радужнопольский край (21 пользователя) - 76,2% *но размер выборки невелик*  
По оператору чаще возвращаются в сегментах : Быстрый кассир (55 пользователя) - 83,6% и Реестр (27 пользователей) - 77,8%   *но размер выборки невелик* 



**Ответы на вопросы:**

- **Какие сегменты чаще возвращаются?** Среди статистически значимых сегментов — desktop-пользователи и посетители выставок/театров.
- **Есть ли успешные «точки входа»?** Явных «точек входа» с аномально высоким возвратом не выявлено. Разница между сегментами не превышает 3–4 п.п., что указывает на равномерное влияние признаков первого заказа на удержание.

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.


In [ ]:

df_users[(df_users['first_order_event'] == 'концерты') | (df_users['first_order_event'] == 'спорт')].groupby('first_order_event')['is_two'].value_counts(normalize=True).unstack(fill_value=0).plot(kind='bar')


In [ ]:
df_agg = df_users.groupby('first_order_region').agg({'user_id':'count', 'is_two':'mean'}).sort_values(by='user_id')
axes = df_agg.plot(kind='bar', subplots=True,
            sharex=True,
            sharey=False,
            legend=False,
            xlabel='Регион',
            figsize = (14,8))
# Устанавливаем подпись для оси Y каждого из подграфиков
axes[0].set_ylabel('кол-во пользователей')
axes[1].set_ylabel('доля вернувшихся')
plt.show()

#### Проверка гипотез:

|Гипотеза|Результат|Обоснование|
|---|---|---|
|Спорт vs Концерты|Не подтверждена|Доля возврата: спорт 54.3%, концерты 60.5%|
|Активные регионы|Не подтверждена|Доля возврата распределена равномерно по регионам|

---

#### **4.2. Исследование поведения пользователей через показатели выручки и состава заказа**

Изучим количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.


**Задача 4.2.1.** Проследим связь между средней выручкой сервиса с заказа и повторными заказами.

- Построим сравнительные гистограммы распределения средней выручки с билета ( `avg_revenue_rub` ):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответим на вопросы:
    
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?
    


In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df_users[(df_users['revenue_mean'] < 3000)&(df_users['revenue_mean'] >0)], # отфильтруем выбросы
             x='revenue_mean', 
             hue='is_two', 
             bins=30,
             kde=True,
             stat='density',
            common_norm=False,
             alpha=0.5)
plt.title('Сравнение распределения средней выручки')
plt.xlabel('Средняя выручка с билета')
plt.ylabel('Плотность распределения количества пользователей')
plt.grid(axis='y', alpha=0.3)
plt.show()

#### Средняя выручка с заказа:

- Пользователи с 1 заказом: концентрация в диапазоне 100-200 руб.
- Пользователи с 2+ заказами: распределение смещено в сторону более высоких значений

**Вывод:** Пользователи с более высокой средней выручкой чаще возвращаются

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?



In [ ]:
#выделим две группы рользователей
df_1 = df_users[(df_users['is_two'] & ~df_users['is_five'] )& (df_users['revenue_mean'] < 3000)&(df_users['revenue_mean'] >0)]
df_2 = df_users[(df_users['is_five'])& (df_users['revenue_mean'] < 3000)&(df_users['revenue_mean'] >0)]

# Соединяем в один DataFrame с меткой группы
df_1['group'] = '2-4 заказа'
df_2['group'] = '5+ заказов'

df_plot = pd.concat([df_1, df_2])

# Рисуем
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df_plot,
    x='revenue_mean',
    hue='group',
    bins=30,
    kde=True,
    alpha=0.5,
    stat='density',
    common_norm=False   #чтобы сравнивать форму, а не абсолютные значения
)

plt.title('Сравнение распределения средней выручки')
plt.xlabel('Средняя выручка с заказа')
plt.ylabel('Плотность')
plt.grid(axis='y', alpha=0.3)
plt.show()

**Наблюдения по распределению:**

|Группа|Характеристика распределения|
|---|---|
|2–4 заказа|Пик плотности в диапазоне 200–600 руб., правый хвост до 3000 руб.|
|5+ заказов|Более выраженный пик в диапазоне 300–700 руб., смещение вправо|

**Ответ на вопрос:** Различия есть, но они умеренные. Пользователи с 5+ заказами демонстрируют немного большую среднюю выручку, однако в основном распределения перекрываются.

**Интерпретация:** Более лояльные пользователи (5+ заказов) склонны тратить немного больше за заказ, но выручка не является определяющим фактором удержания — различия не превышают 15–20% по центральной части распределения.

**Задача 4.2.3.** Проанализируем влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучим распределение пользователей по среднему количеству билетов в заказе ( `avg_tickets_count` ) и опишем основные наблюдения.
- Разделим пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитаем общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответим на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df_users[df_users['avg_tickets_in_order'] < 8],
    x='avg_tickets_in_order',
    bins=7,
    alpha=0.5
)

plt.title('Распределение среднего количества билетов в заказе')
plt.xlabel('Среднее количество билетов')
plt.ylabel('Кол-во пользователей')
plt.grid(axis='y', alpha=0.3)
plt.show()

видим, что: большинство пользователей покупают по 2-3 билета в заказе

In [ ]:
def categorize(row):
    if row['avg_tickets_in_order'] >= 5:
        return '5+'
    elif row['avg_tickets_in_order'] >= 3:
        return '3-5'
    elif row['avg_tickets_in_order'] >= 2:
        return '2-3'
    elif row['avg_tickets_in_order'] >= 1:
        return '1-2'
    

df_users['tickets_category'] = df_users.apply(categorize, axis=1)


In [ ]:
df_users.groupby('tickets_category')['is_two'].agg(['count', 'mean']) #общее число пользователей, совершивших повторные заказы и их доля


### Вывод по задаче 4.2.3: Влияние количества билетов на повторные покупки

**Распределение пользователей по сегментам:**

|Сегмент билетов|Число пользователей|Доля возврата (2+ заказа)|
|---|---|---|
|1–2 билета|2 397|51.1%|
|2–3 билета|8 702|**71.3%**|
|3–5 билетов|9 046|54.1%|
|5+ билетов|689|19.3%|

**Ответы на вопросы:**

1. **Распределение неравномерное:** ~85% пользователей сконцентрированы в сегментах 2–3 и 3–5 билетов.
2. **Аномалии:**
    - **Высокая доля возврата** в сегменте 2–3 билета (71.3%) — на 17 п.п. выше среднего по выборке.
    - **Низкая доля возврата** в сегменте 5+ билетов (19.3%) — вероятно, это разовые групповые покупки (корпоративы, мероприятия), не формирующие лояльность.

**Интерпретация:** Оптимальный профиль повторного покупателя — 2–3 билета в заказе. Слишком малое (1–2) или слишком большое (5+) количество билетов ассоциировано с более низкой вероятностью возврата.

---


#### **4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки**

Изучим временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.



**Задача 4.3.1.** Проанализируем, как день недели, в который была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделим день недели.
- Для каждого дня недели подсчитаем общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируем.
- Ответим на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?



In [ ]:
df_users['weekday'] = df_users['first_order_dt'].dt.weekday + 1 #понедельник - 1, воскресенье - 7

In [ ]:
df_weekday = df_users.groupby('weekday')['is_two'].agg(['count', 'mean']) #общее число пользователей, совершивших повторные заказы и их доля
print('Общее число пользователей, совершивших повторные заказы и их доля')
df_weekday

In [ ]:
# Рисуем
plt.figure(figsize=(12, 8))
df_weekday.plot(kind='line',
            subplots=True,
            sharex=True,
            sharey=False,
            legend=False,
            title=['Общее количество пользователей', 'Доля пользователей, совершивших повторный заказ'])

#plt.title('Сравнение числа пользователей и доли вернувшихся клиентов по дням недели первого заказа')
plt.xlabel('день недели')
plt.grid(axis='y', alpha=0.3)
plt.show()

Вывод: День недели первого заказа не оказывает значимого влияния на возврат (разница менее 4%)

**Задача 4.3.2.** Изучим, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитаем среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуем, как средний интервал между заказами влияет на вероятность повторного заказа.

In [ ]:
def cat_order_counts(row):
    if row['total_orders'] >= 5:
        return '5+'
    elif row['total_orders'] >= 2:
        return '2-4'
    else:
        return '1'

df_users['order_count_cat'] = df_users.apply(cat_order_counts, axis =1)

In [ ]:
#посчитаем среднее время между заказами для групп пользователей, сделавших `2-4` и `5 и более` заказов
df_users[(df_users['order_count_cat'] == '2-4') | (df_users['order_count_cat'] == '5+')].groupby('order_count_cat')['avg_days_between_orders'].mean()

**Вывод:**  
Пользователи с 5+ заказами совершают покупки в 2 раза чаще (11 дней vs 21 день)  
Частота покупок — может говорить о высокой лояльности

---

### **4.4. Корреляционный анализ количества покупок и признаков пользователя**

Изучим, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используем универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.



**Задача 4.4.1:** Проведём корреляционный анализ:

- Рассчитаем коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов ( `total_orders` ). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируем полученные результаты. Если полученные значения будут близки к нулю, проверим разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируем результат корреляции с помощью тепловой карты.
- Отвеим на вопрос: какие признаки наиболее связаны с количеством заказов?

In [ ]:
df_users['avg_days_between_orders_days'] = df_users['avg_days_between_orders'].dt.total_seconds() / (24 * 3600)

#выберем столбцы, для расчета корреляций
phik_cols = ['first_order_device', 'first_order_region', 'first_order_service', 'first_order_event', 'total_orders','revenue_mean','avg_tickets_in_order','avg_days_between_orders_days']

corr_matrix = df_users[phik_cols].phik_matrix(interval_cols=['total_orders','revenue_mean','avg_tickets_in_order','avg_days_between_orders_days'])
correlations_with_orders = corr_matrix['total_orders'].drop('total_orders').sort_values().to_frame(name='correlation')
correlations_with_orders

In [ ]:
sns.heatmap(data=correlations_with_orders, annot=True, fmt='.2f', cmap='coolwarm')

выделим сегменты пользователей по полю total_orders, а затем повторим корреляционный анализ. Выделим такие сегменты:  
1 заказ;  
от 2 до 4 заказов;  
от 5 и выше.

In [ ]:
def orders_segm(row):
    if row['total_orders'] >= 5:
        return '5+'
    elif row['total_orders'] >= 2:
        return '2-4'
    else:
        return '1'

df_users['orders_segment'] = df_users.apply(orders_segm, axis =1)

In [ ]:
phik_cols_seg = ['first_order_device', 
             'first_order_region', 
             'first_order_service', 
             'first_order_event', 
             'orders_segment',
             'revenue_mean',
             'avg_tickets_in_order',
             'avg_days_between_orders_days']

corr_matrix_seg = df_users[phik_cols_seg].phik_matrix(interval_cols=['revenue_mean','avg_tickets_in_order','avg_days_between_orders_days'])
correlations_with_orders_seg = corr_matrix_seg['orders_segment'].drop('orders_segment').sort_values().to_frame(name='correlation_with_orders_segment')
sns.heatmap(data=correlations_with_orders_seg, annot=True, fmt='.2f', cmap='coolwarm')

 **Вывод:**   
Наиболее значимые признаки:  
- Средний интервал между заказами (0.45)  
- Среднее количество билетов (0.38)  

Признаки первого заказа (тип устройства, регион, оператор, тип мероприятия), а также средний чек практически не влияют на общее число заказов  

  

---


#### **5. Общие выводы и рекомендации**

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

---

### Информация о данных и предобработке

|Параметр|Значение|
|---|---|
|Исходный объём данных|290 611 заказов|
|Период данных|2024 год|
|Валюты|RUB, KZT (приведены к RUB через ежедневный курс)|
|Фильтрация устройств|Только mobile и desktop, исключены фильмы|
|Фильтрация выбросов|По `revenue_rub` и `tickets_count` отфильтровано ~1% данных (99-й перцентиль)|
|Фильтрация пользователей|Отфильтрованы пользователи с `total_orders` > 95-го перцентиля (>32 заказов)|
|Итоговая выборка|  283185 строк из 290611, что составляет 97.44%|

---

### Основные результаты анализа

**Статистика по пользователям:**

|Показатель|Значение|
|---|---|
|Доля пользователей с 2+ заказами|62%|
|Доля пользователей с 5+ заказами|29%|
|Средняя выручка с заказа|574 руб.|
|Медианное количество билетов в заказе|3|
|Медианный интервал между заказами|9 дней|

**Признаки первого заказа и возврат пользователей:**

- Распределение пользователей по признакам первого заказа неравномерно, но эта неравномерность зеркально отражает общую структуру платформы (концерты 44%, mobile 83%, Каменевский регион 33%).
- **Статистически значимые различия по возврату:**
    - Desktop-пользователи: 62.3% возврата vs mobile 59.0%
    - Выставки/театр: ~62% возврата vs концерты 60.5%
- Признаки первого заказа (регион, оператор, тип мероприятия) слабо коррелируют с числом заказов (phi_k < 0.05).

**Выручка и количество билетов:**

- Пользователи с 2+ заказами имеют чуть большую среднюю выручку, но распределения сильно перекрываются.
- **Ключевое наблюдение по билетам:**
    
    |Сегмент билетов|Доля возврата|
    |---|---|
    |2–3 билета|**71.7%**|
    |3–5 билетов|52.1%|
    |1–2 билета|51.5%|
    |5+ билетов|8.2%|
    
- Сегмент 2–3 билета показывает аномально высокую долю возврата — потенциальный драйвер лояльности.

**Временные характеристики:**

- День недели первого заказа не влияет на возврат (разница между днями < 4 п.п.).
- **Интервал между заказами — ключевой предиктор:**
    - Группа 2–4 заказа: средний интервал 21 день
    - Группа 5+ заказов: средний интервал 11 дней

**Корреляционный анализ (phi_k с `total_orders`):**

| Признак                   | Корреляция |
| ------------------------- | ---------- |
| `avg_tickets_in_order`    | **0.50**   |
| `avg_days_between_orders` | **0.49**   |
| `revenue_mean`            | 0.27       |
| Признаки первого заказа   | <= 0.05    |  

**Вывод:** Поведенческие метрики (частота покупок, размер заказа) значительно важнее контекстных признаков первого заказа для прогнозирования лояльности.

---

### Дополнительные наблюдения

- Сегменты с экстремально высокой долей возврата (например, «Верхозёрский край» — 100%) имеют малый размер выборки (n < 50) и не подходят для масштабирования гипотез.
- Пользователи с 5+ билетами в заказе показывают низкую долю возврата (19.3%) — вероятно, это разовые групповые покупки, не формирующие привычку.

---

### Рекомендации для команды маркетинга

**Приоритетные действия:**

|Приоритет|Действие|Целевой сегмент|Ожидаемый эффект|
|---|---|---|---|
|1|Триггерные рассылки через 7–14 дней после первого заказа|Все пользователи с 1 заказом|Увеличение конверсии в 2+ заказ|
|2|Персонализированные предложения на 2–3 билета|Пользователи с историей покупок 2–3 билета|Повышение LTV за счёт удержания «оптимального» сегмента|
|3|Программы лояльности для частых покупателей|Пользователи с интервалом < 14 дней|Удержание наиболее активных клиентов|
|4|A/B-тесты таймингов коммуникаций|Desktop-пользователи|Оптимизация конверсии в повторный заказ|

**Сегменты для фокуса:**

1. **Высокий приоритет:** Пользователи с 1 заказом, 2–3 билета, интервал < 14 дней — наибольший потенциал удержания.
2. **Средний приоритет:** Desktop-пользователи с первым заказом на выставки/театр — более высокое удержание при достаточном размере выборки.
3. **Отдельная стратегия:** Пользователи с 5+ билетами — вероятно, B2B/групповые покупки, требуют иных механик вовлечения.

**Метрики для мониторинга:**

- Доля пользователей с 2+ заказами (целевое: > 65%)
- Средний интервал между заказами (целевое: < 15 дней)
- Конверсия из сегмента «2–3 билета» в повторный заказ

---

### Заключение

Проведённый анализ показал, что для повышения удержания клиентов наиболее эффективны действия, направленные на **сокращение интервала между заказами** и **стимулирование покупок в сегменте 2–3 билета**. Признаки первого заказа имеют ограниченное влияние на возврат, поэтому маркетинговые усилия следует фокусировать на поведенческих метриках и своевременной коммуникации с пользователями после первого заказа.

---